# EA1 - Actividad 1.1: Introduccion a Apache Spark

## Objetivos
- Entender que es Apache Spark y por que es relevante en Big Data
- Crear y configurar una SparkSession
- Leer y explorar datos basicos desde un archivo CSV
- Realizar operaciones fundamentales: select, filter, groupBy

## Conceptos Clave

### Que es Apache Spark?

Apache Spark es un **motor de procesamiento distribuido** de datos a gran escala, disenado para ser rapido y de proposito general.

**Breve historia:**
- Nacio en 2009 en el AMPLab de UC Berkeley
- Donado a la Apache Software Foundation en 2013
- Hoy es uno de los proyectos open-source mas activos en Big Data

**Ventajas sobre MapReduce tradicional:**
- **Velocidad:** Procesamiento en memoria (hasta 100x mas rapido que MapReduce en disco)
- **Facilidad de uso:** APIs en Python, Scala, Java y R
- **Versatilidad:** Batch, streaming, ML, grafos y SQL en un solo framework
- **Evaluacion lazy:** Las transformaciones no se ejecutan hasta que se necesita un resultado

**Ecosistema Spark:**

| Componente | Descripcion |
|------------|-------------|
| Spark Core | Motor base de procesamiento distribuido |
| Spark SQL | Consultas estructuradas y DataFrames |
| Spark Streaming | Procesamiento de datos en tiempo real |
| MLlib | Libreria de Machine Learning distribuido |
| GraphX | Procesamiento de grafos |

## Arquitectura de Spark

```
                    +-------------------+
                    |   Aplicacion      |
                    |   (tu codigo)     |
                    +--------+----------+
                             |
                    +--------v----------+
                    |   SparkSession    |
                    |   (Driver)        |
                    +--------+----------+
                             |
                    +--------v----------+
                    | Cluster Manager   |
                    | (YARN/Mesos/      |
                    |  Standalone)      |
                    +---+----------+----+
                        |          |
               +--------v--+  +----v-------+
               | Executor 1|  | Executor 2 |
               | [Task]    |  | [Task]     |
               | [Task]    |  | [Task]     |
               +-----------+  +------------+
```

- **Driver:** Programa principal que coordina la ejecucion
- **SparkSession:** Punto de entrada unificado (reemplaza SparkContext, SQLContext, HiveContext)
- **Cluster Manager:** Asigna recursos (YARN, Mesos, Kubernetes o Standalone)
- **Executors:** Procesos que ejecutan las tareas en los nodos del cluster

In [ ]:
# Setup: Crear SparkSession
# La SparkSession es el punto de entrada principal para trabajar con Spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01_introduccion_spark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Leer datos desde un archivo CSV

Spark puede leer datos de multiples formatos: CSV, JSON, Parquet, ORC, JDBC, etc.

Parametros importantes de `spark.read.csv()`:
- `header=True`: La primera fila contiene los nombres de las columnas
- `inferSchema=True`: Spark infiere automaticamente los tipos de datos

In [ ]:
# Leer el archivo flights.csv
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True, nullValue="NA")

# Verificamos que se leyo correctamente
print(f"Tipo del objeto: {type(df)}")

## Explorar los datos

Una vez cargado el DataFrame, tenemos varias formas de explorarlo.

In [ ]:
# Mostrar las primeras 5 filas del DataFrame
df.show(5)

In [ ]:
# Ver el esquema (nombres de columnas y tipos de datos)
df.printSchema()

In [ ]:
# Contar el numero total de filas
print(f"Numero total de filas: {df.count()}")

# Ver los nombres de las columnas
print(f"Columnas: {df.columns}")

In [ ]:
# Estadisticas descriptivas basicas (media, desviacion estandar, min, max, count)
df.describe().show()

## Seleccionar columnas con `select()`

El metodo `select()` permite elegir columnas especificas del DataFrame, similar a SELECT en SQL.

In [ ]:
# Ejemplo: seleccionar solo las columnas de aerolinea y retraso
df.select("carrier", "dep_delay", "arr_delay").show(5)

## Filtrar datos con `filter()`

El metodo `filter()` permite seleccionar filas que cumplan una condicion, similar a WHERE en SQL.

In [ ]:
# Ejemplo: filtrar vuelos con distancia mayor a 2000 millas
vuelos_largos = df.filter(df["distance"] > 2000)
print(f"Vuelos de larga distancia (>2000 mi): {vuelos_largos.count()}")
vuelos_largos.show(5)

## Agrupar datos con `groupBy()`

El metodo `groupBy()` agrupa los datos por una o mas columnas y permite aplicar funciones de agregacion como `count()`, `sum()`, `avg()`, etc.

In [ ]:
# Ejemplo: contar vuelos por mes
df.groupBy("month").count().orderBy("month").show()

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Seleccionar columnas especificas
# =============================================================
# Seleccionar las columnas de aeropuerto origen, destino y retraso de salida

df_seleccion = df.select("origin", "dest", "dep_delay")
df_seleccion.show(10)

> **Nota docente:** El metodo `select()` es analogo al SELECT de SQL. Los estudiantes suelen cometer el error de pasar los nombres de columnas como una lista (`select(["col1", "col2"])`) en lugar de argumentos separados (`select("col1", "col2")`). Ambas formas funcionan, pero la segunda es la mas comun. Tambien es valido usar `select(df.origin, df.dest, df.dep_delay)` o `select(F.col("origin"), ...)`. Aprovechar para explicar que `select()` retorna un nuevo DataFrame (inmutabilidad).

In [ ]:
# =============================================================
# EJERCICIO 2: Filtrar vuelos con retraso
# =============================================================
# Filtrar vuelos que salieron con retraso (dep_delay > 0)
# dep_delay ya es IntegerType gracias a nullValue="NA" en la lectura

df_retrasados = df.filter(df["dep_delay"] > 0)
print(f"Vuelos con retraso: {df_retrasados.count()}")
print(f"Porcentaje: {df_retrasados.count() / df.count() * 100:.1f}%")

> **Nota docente:** El metodo `filter()` acepta tanto la sintaxis `df["col"] > valor` como `F.col("col") > valor`. Error comun: los estudiantes olvidan que `filter()` no modifica el DataFrame original sino que retorna uno nuevo. Tambien es frecuente confundir `>= 0` con `> 0` (un retraso de 0 minutos significa que salio a tiempo, no con retraso). Gracias a `nullValue="NA"` en la lectura, `dep_delay` ya es IntegerType y los valores "NA" son null, por lo que no se necesita cast manual. El calculo del porcentaje llama a `.count()` dos veces, lo cual ejecuta dos acciones sobre el cluster; en produccion seria mejor guardar los conteos en variables para evitar recalculos.

In [ ]:
# =============================================================
# EJERCICIO 3: Contar vuelos por aerolinea
# =============================================================
# Agrupar por aerolinea, contar y ordenar de mayor a menor

df_aerolineas = df.groupBy("carrier").count().orderBy("count", ascending=False)
df_aerolineas.show()

> **Nota docente:** `groupBy().count()` es el patron mas simple de agregacion. El metodo `orderBy()` (alias de `sort()`) acepta el parametro `ascending=False` para orden descendente. Error comun: los estudiantes escriben `ascending=false` (minuscula) en vez de `ascending=False`. Notar que la columna generada por `.count()` se llama literalmente "count"; si se desea renombrar, se puede usar `.agg(F.count("*").alias("total_vuelos"))` en lugar de `.count()`.

---
## Resumen

En esta actividad aprendimos:

1. **Que es Apache Spark:** Un motor de procesamiento distribuido, rapido y versatil
2. **Arquitectura:** Driver, Executors, Cluster Manager y SparkSession como punto de entrada
3. **Crear SparkSession:** `SparkSession.builder.appName(...).master(...).getOrCreate()`
4. **Leer CSV:** `spark.read.csv(path, header=True, inferSchema=True)`
5. **Explorar datos:** `.show()`, `.printSchema()`, `.count()`, `.columns`, `.describe()`
6. **Operaciones basicas:** `.select()`, `.filter()`, `.groupBy()`, `.orderBy()`

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Encuentra el **top 5 de rutas (origen-destino) mas frecuentes**.

Pista: Puedes agrupar por dos columnas a la vez con `groupBy("col1", "col2")`.

In [ ]:
# =============================================================
# DESAFIO: Top 5 rutas mas frecuentes
# =============================================================
# Agrupar por origen y destino, contar y mostrar las 5 mas frecuentes

df_rutas = df.groupBy("origin", "dest") \
    .count() \
    .orderBy("count", ascending=False)
df_rutas.show(5)

> **Nota docente:** Este ejercicio demuestra que `groupBy()` acepta multiples columnas, generando grupos unicos por cada combinacion (origen, destino). Punto de discusion interesante: la ruta SFO->LAX y LAX->SFO se cuentan como rutas diferentes. Si quisieramos tratarlas como la misma ruta (ida y vuelta), habria que normalizar creando columnas auxiliares con `F.least()` y `F.greatest()`. Este patron de "top N" es muy comun en analisis exploratorio.

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")